# VD AI — Генератор на GPU (Google Colab)
Генерирует изображения (SD 1.5) и видео (Text2Video) на GPU T4

In [ ]:
# 1. Установка пакетов
!pip install diffusers transformers accelerate torch torchvision pillow gradio opencv-python -q
print("Готово")

In [ ]:
# 2. Загрузка модели изображений (SDXL-turbo — быстрая и умная)
import torch
from diffusers import AutoPipelineForText2Image

device = "cuda"
img_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    use_safetensors=True,
)
img_pipe = img_pipe.to(device)
print("Модель изображений готова (SDXL-turbo)!")

In [ ]:
# 3. Простая генерация изображения (быстро — 2 шага!)
# Меняй prompt и запускай эту ячейку
prompt = "обезьяна в джунглях, крупным планом, высокое качество"
image = img_pipe(
    prompt=prompt,
    width=512, height=512,
    guidance_scale=0,
    num_inference_steps=2,
    generator=torch.Generator(device=device).manual_seed(42),
).images[0]
display(image)
image.save("result.jpg")
print(f"Готово: {prompt}")

In [ ]:
# 4. ЗАГРУЗКА МОДЕЛИ ВИДЕО (Text-to-Video)
from diffusers import TextToVideoSDPipeline

vid_pipe = TextToVideoSDPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16",
)
vid_pipe = vid_pipe.to(device)
vid_pipe.enable_attention_slicing()
print("Модель видео готова!")

In [ ]:
# 5. Генерация видео
# Меняй prompt и запускай
from diffusers.utils import export_to_video

prompt = "медведь играет на гитаре"
result = vid_pipe(
    prompt=prompt,
    num_frames=16,
    num_inference_steps=25,
    width=512, height=512,
    generator=torch.Generator(device=device).manual_seed(42),
)

export_to_video(result.frames[0], "video_result.mp4", fps=8)
from IPython.display import Video
display(Video("video_result.mp4", width=400))
print(f"Видео готово: {prompt}")

In [ ]:
# 6. Веб-интерфейс (Gradio) — изображения и видео
import gradio as gr

def generate_image(prompt, steps, guidance, seed):
    gen = torch.Generator(device=device).manual_seed(seed) if seed != 0 else None
    result = img_pipe(
        prompt=prompt,
        width=768, height=768,
        guidance_scale=guidance,
        num_inference_steps=steps,
        generator=gen,
    )
    return result.images[0]

def generate_video(prompt, steps, seed):
    gen = torch.Generator(device=device).manual_seed(seed) if seed != 0 else None
    result = vid_pipe(
        prompt=prompt,
        num_frames=16,
        num_inference_steps=steps,
        width=512, height=512,
        generator=gen,
    )
    path = "output_video.mp4"
    export_to_video(result.frames[0], path, fps=8)
    return path

with gr.Blocks(title="VD AI") as demo:
    gr.Markdown("# VD AI — Генерация на GPU (T4)")
    with gr.Tab("Изображение"):
        with gr.Row():
            with gr.Column():
                img_prompt = gr.Textbox(label="Промпт", value="обезьяна в джунглях, крупным планом, высокое качество")
                img_steps = gr.Slider(1, 8, value=2, label="Шаги (1-4 быстро, 8 качественно)")
                img_guidance = gr.Slider(0, 5, value=0, label="Guidance")
                img_seed = gr.Number(value=0, label="Seed")
                img_btn = gr.Button("Сгенерировать изображение")
            with gr.Column():
                img_out = gr.Image(label="Результат")
        img_btn.click(generate_image, [img_prompt, img_steps, img_guidance, img_seed], img_out)
    
    with gr.Tab("Видео"):
        with gr.Row():
            with gr.Column():
                vid_prompt = gr.Textbox(label="Промпт", value="медведь играет на гитаре")
                vid_steps = gr.Slider(5, 50, value=25, label="Шаги")
                vid_seed = gr.Number(value=0, label="Seed")
                vid_btn = gr.Button("Сгенерировать видео")
            with gr.Column():
                vid_out = gr.Video(label="Результат")
        vid_btn.click(generate_video, [vid_prompt, vid_steps, vid_seed], vid_out)

demo.launch(share=True)
print("ССЫЛКА ВЫШЕ — открой её в браузере")